<a href="https://colab.research.google.com/github/qk8015-lgtm/114-2-ProgramingLanguage/blob/main/HW2_%E6%88%90%E7%B8%BE%E4%B8%80%E6%9C%AC%E9%80%9A_part1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
!pip install -q google-generativeai

In [9]:
import gradio as gr
import pandas as pd
from google.colab import auth
from google.auth import default

# -*- coding: utf-8 -*-
import gspread
from datetime import datetime
import os
import json
import google.api_core.exceptions

In [10]:
from google.colab import userdata
import google.generativeai as genai

# 從 Colab Secrets 中獲取 API 金鑰
api_key = userdata.get('gemini')

# 使用獲取的金鑰配置 genai 模組
genai.configure(api_key=api_key)

MODEL_ID = 'gemini-2.5-flash'

In [11]:
model_instance = genai.GenerativeModel(model_name=MODEL_ID)
response = model_instance.generate_content(
    contents="Explain how AI works in a few words"
)
print(response.text)

AI learns patterns from data to make decisions or predictions.


In [12]:
import gradio as gr
import pandas as pd
from google.colab import auth
from google.auth import default

# -*- coding: utf-8 -*-
import gspread
from datetime import datetime
import os
import json
from google.colab import userdata
import google.generativeai as genai # Corrected import to google.generativeai

# 從 Colab Secrets 中獲取 API 金鑰
api_key = userdata.get('gemini')

# 使用獲取的金鑰配置 genai 模組
genai.configure(api_key=api_key)

MODEL_ID = 'gemini-2.5-flash'

SHEET_URL = "https://docs.google.com/spreadsheets/d/1sa2n02HTV0YGSzoM4XJyjpj4pm5tklrC76p6okxCveY/edit?usp=sharing"
WORKSHEET_NAME = "工作表2"

REQUIRED_COLUMNS = ["日期", "科目", "作業成績"]

_auth_done = False
_gc = None
_ws = None

# --- 主要功能區塊 ---
def get_user_grades():
    """
    透過終端機輸入學生成績，直到使用者輸入 'q' 結束。
    """
    print("--- 準備輸入成績。輸入 'q' 來停止。---")
    grades = []
    while True:
        print("\n" + "="*30) # Add a separator before each new entry
        subject = input("請輸入科目 (輸入 'q' 停止): ")
        if subject.lower() == 'q':
            break

        grade_input = input(f"  請輸入 {subject} 的成績: ")
        try:
            grade = int(grade_input)
        except ValueError:
            print("  成績必須是數字。請重新輸入。")
            continue

        today = datetime.now().strftime('%Y-%m-%d')
        grades.append([today, subject, grade])
        print(f"  ✔ 已記錄：日期 {today}, 科目 {subject}, 成績 {grade}\n")

    return grades

def remove_symbols(text, symbols_to_remove=None):
    """
    移除字串中指定的符號。
    預設移除 Markdown 標題和列表符號。
    """
    if symbols_to_remove is None:
        # 移除常用的 Markdown 標題和列表符號，以及分隔線符號
        symbols_to_remove = ['#', '*', '-']

    for symbol in symbols_to_remove:
        text = text.replace(symbol, '')
    return text.strip()

def get_ai_summary(grades):
    """
    呼叫 Gemini 模型來生成成績摘要與常見迷思。
    """
    # 準備給 AI 的提示
    prompt_text = "以下是學生的成績列表，請幫我根據這些成績，產出一個簡單的摘要與常見迷思整理（不評分，只做總結）。\n\n"
    for record in grades:
        date, subject, grade = record
        prompt_text += f"日期：{date}, 科目：{subject}, 成績：{grade}\n"

    print("\n--- 正在呼叫 AI 模型生成摘要... ---")
    try:
        # Updated generate_content call
        response = genai.GenerativeModel(model_name=MODEL_ID).generate_content(contents = prompt_text)
        summary = response.text
        # 移除摘要中的符號
        summary = remove_symbols(summary)
        # 保持原始換行符號，讓摘要能在 Google Sheet 的單一儲存格中分行顯示。
        return summary
    except Exception as e:
        print(f"呼叫 AI 時發生錯誤：{e}")
        return "AI 摘要生成失敗。"

def main():
    """
    主程式流程：輸入成績 -> 獲取 AI 摘要 -> 寫入 Google Sheet。
    """
    try:
        # 1. Google Sheet 身份驗證
        auth.authenticate_user()

        creds, _ = default()
        gc = gspread.authorize(creds)

        sh = gc.open_by_url(SHEET_URL)
        ws = sh.worksheet(WORKSHEET_NAME)

        print("--- Google Sheet 連線成功。---")

        # 2. 獲取使用者輸入的成績
        new_grades = get_user_grades()

        if not new_grades:
            print("沒有輸入任何成績，程式結束。")
            return

        # 3. 獲取 AI 摘要
        summary = get_ai_summary(new_grades)

        # 4. 準備所有要寫入 Google Sheet 的行
        all_rows_to_append = []

        # 將新成績加入列表 (順序調整：先成績)
        for grade_record in new_grades:
            all_rows_to_append.append(grade_record)

        all_rows_to_append.append(['', '', '']) # 加入一個空行作為分隔

        # 生成代表剛登錄成績的字串
        grade_summary_text = ""
        if new_grades:
            grade_entries = [f"{subject}:{grade}分" for _, subject, grade in new_grades]
            grade_summary_text = "登錄成績: " + ", ".join(grade_entries)
        else:
            grade_summary_text = "無新成績登錄"

        # 將 AI 摘要加入列表 (統整到一個不要分很多列，且在成績下方)
        # 變更：將第一列的日期改為實際成績概述
        all_rows_to_append.append([grade_summary_text, 'AI 摘要', summary if summary else '無摘要內容'])

        # 5. 將所有資料一次性寫入 Google Sheet
        ws.append_rows(all_rows_to_append)
        print("\n--- AI 摘要與成績已成功寫入 Google Sheet。---")

        # 在 Colab 中顯示 AI 生成的摘要內容
        print("以下是 AI 生成的摘要內容：")
        print("-" * 50)
        print(summary)
        print("-" * 50)

    except gspread.exceptions.APIError as e:
        print(f"Google Sheets API 錯誤：{e.response.text}")
        print("請確認：")
        print("1. 您的服務帳戶金鑰檔案正確且未過期。")
        print("2. 您已將服務帳戶的 Email 地址（在 JSON 檔案中）分享給 Google Sheet，並給予編輯權限。")
    except Exception as e:
        print(f"發生未預期的錯誤：{e}")

if __name__ == "__main__":
    main()

發生未預期的錯誤：Error: credential propagation was unsuccessful


In [13]:
main()

--- Google Sheet 連線成功。---
--- 準備輸入成績。輸入 'q' 來停止。---

請輸入科目 (輸入 'q' 停止): 國文ˊ
  請輸入 國文ˊ 的成績: 55
  ✔ 已記錄：日期 2026-04-18, 科目 國文ˊ, 成績 55


請輸入科目 (輸入 'q' 停止): q

--- 正在呼叫 AI 模型生成摘要... ---

--- AI 摘要與成績已成功寫入 Google Sheet。---
以下是 AI 生成的摘要內容：
--------------------------------------------------
好的，這是一份根據您提供的成績資料所做的簡單摘要與常見迷思整理，不帶任何評分或判斷。



 學生單次成績摘要

根據您提供的資料：
   科目： 國文
   成績： 55分
   日期： 2026年4月18日

總結：
這位學生在2026年4月18日的國文科目中，取得了一次55分的成績。這個分數是針對單一科目和單一時間點的學習成果展現。



 關於成績的常見迷思整理

成績在教育體系中扮演著重要角色，但人們對成績的解讀常存在一些迷思。以下是幾個常見的迷思及其澄清：

1.  迷思一：成績是衡量學生能力和未來成功的唯一標準。
       澄清： 成績確實反映了學生在特定時間點對特定知識的掌握程度和應試能力，是學習成果的指標之一。然而，它無法完全反映一個人的學習潛力、創造力、解決問題的能力、人際關係、情商、毅力等多元素質，也非預測未來成功的唯一指標。許多在學業上並非頂尖的人，在職場或社會上也能取得巨大成就。

2.  迷思二：低分代表學生不夠努力或不聰明。
       澄清： 低分可能由多種原因造成，例如對特定概念理解不足、學習方法不適合、考試焦慮、時間管理不佳、教材難度、教師教學風格、甚至是當下的身體狀況或家庭壓力等。它通常是學習過程中需要關注的信號，而非對個人智力或品格的判斷。

3.  迷思三：高分就代表完全掌握所有知識。
       澄清： 高分確實顯示學生在特定評估中表現良好，可能代表他們對教材內容理解透徹或應試技巧高超。但有時高分也可能來自於死記硬背，而非真正理解或融會貫通。真正的知識掌握和應用能力，需要更深入的理解、批判性思考和在不同情境下的實踐來驗證。

4.  迷思四：成績是完全客觀且

In [14]:
print('--- Listing available models ---')
for model in genai.list_models():
    if 'generateContent' in model.supported_generation_methods:
        print(f"Model Name: {model.name}, Supported: {model.supported_generation_methods}")

--- Listing available models ---
Model Name: models/gemini-2.5-flash, Supported: ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
Model Name: models/gemini-2.5-pro, Supported: ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
Model Name: models/gemini-2.0-flash, Supported: ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
Model Name: models/gemini-2.0-flash-001, Supported: ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
Model Name: models/gemini-2.0-flash-lite-001, Supported: ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
Model Name: models/gemini-2.0-flash-lite, Supported: ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
Model Name: models/gemini-2.5-flash-preview-tts, Supported: ['countTokens', 'generateContent']
Model Name: models/gemini-2.5-pro-preview-tts, Supported: ['countTokens', 'g

In [15]:
SHEET_URL = "https://docs.google.com/spreadsheets/d/1sa2n02HTV0YGSzoM4XJyjpj4pm5tklrC76p6okxCveY/edit?usp=sharing"
WORKSHEET_NAME = "工作表2"

REQUIRED_COLUMNS = ["日期", "科目", "作業成績"]

_auth_done = False
_gc = None
_ws = None

In [16]:
# --- 主要功能區塊 ---
def get_user_grades():
    """
    透過終端機輸入學生成績，直到使用者輸入 'q' 結束。
    """
    print("--- 準備輸入成績。輸入 'q' 來停止。---")
    grades = []
    while True:
        print("\n" + "="*30) # Add a separator before each new entry
        subject = input("請輸入科目 (輸入 'q' 停止): ")
        if subject.lower() == 'q':
            break

        grade_input = input(f"  請輸入 {subject} 的成績: ")
        try:
            grade = int(grade_input)
        except ValueError:
            print("  成績必須是數字。請重新輸入。")
            continue

        today = datetime.now().strftime('%Y-%m-%d')
        grades.append([today, subject, grade])
        print(f"  ✔ 已記錄：日期 {today}, 科目 {subject}, 成績 {grade}\n")

    return grades

In [23]:
def remove_symbols(text, symbols_to_remove=None):
    """
    移除字串中指定的符號。
    預設移除 Markdown 標題和列表符號。
    """
    if symbols_to_remove is None:
        # 移除常用的 Markdown 標題和列表符號，以及分隔線符號
        symbols_to_remove = ['#', '*', '-']

    for symbol in symbols_to_remove:
        text = text.replace(symbol, '')
    return text.strip()

def get_ai_summary(grades):
    """
    呼叫 Gemini 模型來生成成績摘要與常見迷思。
    """
    # 準備給 AI 的提示
    prompt_text = "以下是學生的成績列表。請直接根據這些成績，產出一個簡單的摘要與常見迷思整理（不評分，只做總結），不要包含任何開頭語句或客套話。\n\n"
    for record in grades:
        date, subject, grade = record
        prompt_text += f"日期：{date}, 科目：{subject}, 成績：{grade}\n"

    # 移除 print 訊息，因為使用者不希望看到步驟
    # print("\n--- 正在呼叫 AI 模型生成摘要... ---")
    try:
        # 使用 genai.GenerativeModel 來生成內容
        response = genai.GenerativeModel(model_name=MODEL_ID).generate_content(contents = prompt_text)
        summary = response.text
        # 移除摘要中的符號
        summary = remove_symbols(summary)
        # 保持原始換行符號，讓摘要能在 Google Sheet 的單一儲存格中分行顯示。
        return summary
    except genai.types.BlockedPromptException as e:
        print(f"API 呼叫失敗：內容被阻擋。錯誤訊息：{e}")
        return "AI 摘要生成失敗：內容被阻擋。"
    except google.api_core.exceptions.GoogleAPICallError as e:
        print(f"API 呼叫失敗：Google API 錯誤。錯誤訊息：{e.message}")
        return "AI 摘要生成失敗：API 服務錯誤。"
    except Exception as e:
        print(f"呼叫 AI 時發生未預期的錯誤：{e}")
        return "AI 摘要生成失敗：未預期錯誤。"

In [18]:
new_grades = get_user_grades()

--- 準備輸入成績。輸入 'q' 來停止。---

請輸入科目 (輸入 'q' 停止): 國文
  請輸入 國文 的成績: 66
  ✔ 已記錄：日期 2026-04-18, 科目 國文, 成績 66


請輸入科目 (輸入 'q' 停止): q


In [19]:
get_ai_summary(new_grades)


--- 正在呼叫 AI 模型生成摘要... ---


'好的，這是一份根據您提供的單一成績所做的簡單摘要與常見迷思整理，完全不帶任何評分判斷。\n\n\n\n 成績摘要\n\n這份資料記錄了學生在2026年4月18日於國文科目中獲得的66分。這是一個單一學科、單一時間點的評量成績紀錄。\n\n\n\n 常見迷思整理（關於成績的解讀）\n\n以下是針對學生成績，特別是單一成績，常見的一些迷思，旨在提供更全面的視角，而非對分數本身進行評價：\n\n1.  迷思一：單一成績能全面定義學生能力。\n       事實： 66分反映的是學生在特定時間點、特定評量（例如：一次考試、一份作業）下的國文表現。它不應被視為學生整體學習能力、學習潛力、個人特質或未來發展的唯一或全面指標。學生在其他科目、其他領域可能展現截然不同的能力。\n\n2.  迷思二：分數高低直接等同於努力程度。\n       事實： 努力通常是成績的影響因素之一，但分數的高低也可能與多種因素相關，例如：學習方法是否有效、是否掌握考試技巧、對特定知識點的理解程度、當天的身心狀態、甚至是評量設計的難易度或評分標準。因此，不能單純地將分數與努力程度劃上等號。\n\n3.  迷思三：一個分數是學生知識掌握的絕對標準。\n       事實： 考試或評量的分數是依據特定的課程內容、出題範圍和評分標準所計算出來的。66分代表學生在該次評量中，對特定內容的掌握程度達到某個水準，但這並不意味著學生對於整個國文科目，或實際語文應用能力的全面掌握程度。評量工具有其局限性，可能無法完全反映學生所有的學習成果。\n\n4.  迷思四：分數代表學習的終點或成功與失敗。\n       事實： 學業成績是學習過程中的一個回饋點，而非學習的終點。它提供了一個檢視學習狀況的機會，可以用來了解哪些方面已經掌握，哪些方面可能需要進一步關注。無論分數如何，學習都是一個持續不斷的過程，每次評量都是成長和調整的機會。'

In [20]:
new_grades

[['2026-04-18', '國文', 66]]

In [21]:
get_ai_summary(new_grades)


--- 正在呼叫 AI 模型生成摘要... ---


'好的，這是一份根據您提供的學生國文成績所做的簡單摘要與常見迷思整理，全程不帶任何評分或判斷：\n\n\n\n 成績摘要\n\n   日期： 2026年4月18日\n   科目： 國文\n   成績： 66分\n   總結： 該學生在2026年4月18日的國文科目成績為66分。此分數為單一科目的表現，數值上落在及格線（通常為60分）之上。\n\n 成績相關常見迷思整理（不評分，只做總結）\n\n1.  迷思一：單一分數即代表學生全部學習能力或努力程度。\n       總結： 這是一個特定時間點、特定科目、特定測驗方式下的表現。它可能受到多種因素影響，例如當天的狀態、測驗範圍、題目難易度、甚至命題方式等，無法全面反映學生的學習投入與掌握程度。\n\n2.  迷思二：成績高低與智力劃上等號。\n       總結： 學術成績是衡量特定學科知識和技能的指標，而智力是一個更廣泛、多面向的概念。許多因素，如學習策略、備考習慣、興趣、甚至心理狀態（如考試焦慮）都可能影響成績，與智力並非簡單的單一對應關係。\n\n3.  迷思三：成績是學習成果的唯一或最重要衡量標準。\n       總結： 除了分數，學生的學習過程中的理解深度、批判性思考能力、解決問題的能力、溝通表達能力、對知識的應用能力，以及學習態度和興趣等，都是重要的學習成果，但這些往往難以透過單一分數來完整呈現。\n\n4.  迷思四：剛好及格（如60幾分）就代表「學得不好」或「有問題」。\n       總結： 及格線以上的成績，代表學生已經達到或超過了該科目設定的基本要求。不同科目、不同測驗難度、甚至不同老師的評分標準都可能讓分數有差異。單看分數高低無法直接判斷學習品質，需要結合更全面的學習表現和教學情境來理解。\n\n5.  迷思五：所有科目的表現都應該一致。\n       總結： 學生在不同科目上往往會有不同的興趣、天賦和學習曲線。單一科目（如國文）的成績，不能直接推論學生在其他科目上的表現，也不能視為其整體學業狀況的代表。'

In [24]:
from datetime import datetime # 將 datetime 匯入語句移到此處以使函式更獨立

def main():
    """
    主程式流程：輸入成績 -> 獲取 AI 摘要 -> 寫入 Google Sheet。
    """
    try:
        # 1. Google Sheet 身份驗證
        auth.authenticate_user()

        creds, _ = default()
        gc = gspread.authorize(creds)

        sh = gc.open_by_url(SHEET_URL)
        ws = sh.worksheet(WORKSHEET_NAME)

        print("--- Google Sheet 連線成功。---")

        # 2. 獲取使用者輸入的成績
        new_grades = get_user_grades()

        if not new_grades:
            print("沒有輸入任何成績，程式結束。")
            return

        # 3. 獲取 AI 摘要
        summary = get_ai_summary(new_grades)

        # 4. 準備所有要寫入 Google Sheet 的行
        all_rows_to_append = []

        # 將新成績加入列表 (順序調整：先成績)
        for grade_record in new_grades:
            all_rows_to_append.append(grade_record)

        all_rows_to_append.append(['', '', '']) # 加入一個空行作為分隔

        # 生成代表剛登錄成績的字串
        grade_summary_text = ""
        if new_grades:
            grade_entries = [f"{subject}:{grade}分" for _, subject, grade in new_grades]
            grade_summary_text = "登錄成績: " + ", ".join(grade_entries)
        else:
            grade_summary_text = "無新成績登錄"

        # 將 AI 摘要加入列表 (統整到一個不要分很多列，且在成績下方)
        # 變更：將第一列的日期改為實際成績概述
        # 根據使用者要求，AI 摘要這行的第一個欄位不顯示 "登錄成績"，改為空白。
        all_rows_to_append.append(['', 'AI 摘要', summary if summary else '無摘要內容'])

        # 5. 將所有資料一次性寫入 Google Sheet
        ws.append_rows(all_rows_to_append)
        print("\n--- AI 摘要與成績已成功寫入 Google Sheet。---")

        # 在 Colab 中顯示 AI 生成的摘要內容
        print("以下是 AI 生成的摘要內容：")
        print("-" * 50)
        print(summary)
        print("-" * 50)

    except gspread.exceptions.APIError as e:
        print(f"Google Sheets API 錯誤：{e.response.text}")
        print("請確認：")
        print("1. 您的服務帳戶金鑰檔案正確且未過期。")
        print("2. 您已將服務帳戶的 Email 地址（在 JSON 檔案中）分享給 Google Sheet，並給予編輯權限。")
    except Exception as e:
        print(f"發生未預期的錯誤：{e}")

if __name__ == "__main__":
    main()

--- Google Sheet 連線成功。---
--- 準備輸入成績。輸入 'q' 來停止。---

請輸入科目 (輸入 'q' 停止): 英文
  請輸入 英文 的成績: 44
  ✔ 已記錄：日期 2026-04-18, 科目 英文, 成績 44


請輸入科目 (輸入 'q' 停止): q

--- AI 摘要與成績已成功寫入 Google Sheet。---
以下是 AI 生成的摘要內容：
--------------------------------------------------
成績摘要
學生的英文科目成績為44分。

常見迷思整理
   迷思一：單一成績代表整體能力。 僅憑單一科目的一次成績，無法全面評估學生在該科目上的整體能力或學習潛力。
   迷思二：成績直接反映努力程度。 成績受多重因素影響，包括考試難度、出題範圍、學生當天的身心狀況、學習方法、甚至突發狀況等，不能直接與努力程度劃上等號。
   迷思三：缺乏比較基準的解讀。 單獨的一個分數，若無班級平均、及格標準、教材內容或考試類型的背景資訊，其意義難以被客觀解讀。
   迷思四：成績是固定的學習標籤。 一次成績只是特定時間點的學習成果展現，學習是一個動態過程，該成績不代表學生未來的表現將一成不變。
--------------------------------------------------
